In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

#### APMの時系列データを取得

In [4]:
apm = 'APM'
market = 'TO'
start = '1999-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

response = request_api.update_stock_timeseries_data(
    code=apm,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [5]:
apm_timeseries_df = request_api.get_stock_time_series_data(
    code=apm,
    market=market,
    start=start,
    end=end
)
apm_timeseries_df

取得件数: 1718


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,upper1,lower1,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1102652,APM,TO,2019-06-27,0.305,0.305,0.220,0.220,62000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1102653,APM,TO,2019-06-28,0.305,0.305,0.305,0.305,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1102654,APM,TO,2019-07-02,0.300,0.300,0.300,0.300,9000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1102655,APM,TO,2019-07-03,0.300,0.300,0.300,0.300,11000,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1102656,APM,TO,2019-07-04,0.300,0.300,0.300,0.300,0,0.285,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1713,1163046,APM,TO,2026-04-27,6.730,6.790,6.580,6.790,204600,6.912,...,7.392999,6.414201,True,NaN,NaN,NaN,NaN,NaN,NaN,False
1714,1163047,APM,TO,2026-04-28,6.470,6.750,6.370,6.680,345700,6.790,...,7.367891,6.519309,False,NaN,6.9436,NaN,NaN,NaN,NaN,False
1715,1163048,APM,TO,2026-04-29,6.330,6.420,6.200,6.350,217900,6.698,...,7.363265,6.538335,False,NaN,NaN,NaN,-0.224748,NaN,NaN,False
1716,1163049,APM,TO,2026-04-30,6.250,6.630,6.250,6.420,339000,6.606,...,7.352761,6.502439,False,NaN,NaN,NaN,NaN,NaN,NaN,False


In [6]:
def stock_prices_and_gold_prices(
        code: str,
        name: str,
        start: str,
        end: str,
        df_sp: pd.DataFrame | None = None,
        df_gold: pd.DataFrame | None = None,
        df_silver: pd.DataFrame | None = None
    ):
    # /api/v1/time_series_data/stock/
    response = request_api.get_stock_time_series_data(
        code=code,
        market=None,
        start=start,
        end=end
    )
    df_stock = pd.DataFrame(response)

    # S&P500を統合
    if df_sp is not None:
        df_sp_tmp = df_sp.copy() if df_sp is not None else pd.DataFrame()
        if "date" not in df_sp_tmp.columns:
            df_sp_tmp = df_sp_tmp.reset_index()

        df_sp_tmp["date"] = pd.to_datetime(df_sp_tmp["date"])
        df_sp_tmp = df_sp_tmp.set_index("date")
        df_sp_tmp = df_sp_tmp.loc[start:end]

    # 金価格を統合
    if df_gold is not None:
        df_gold_tmp = df_gold.copy() if df_gold is not None else pd.DataFrame()
        if "date" not in df_gold_tmp.columns:
            df_gold_tmp = df_gold_tmp.reset_index()
        df_gold_tmp["date"] = pd.to_datetime(df_gold_tmp["date"])
        df_gold_tmp = df_gold_tmp.set_index("date")
        df_gold_tmp = df_gold_tmp.loc[start:end]

    # 銀価格を統合
    if df_silver is not None:
        df_silver_tmp = df_silver.copy() if df_silver is not None else pd.DataFrame()
        if "date" not in df_silver_tmp.columns:
            df_silver_tmp = df_silver_tmp.reset_index()
        df_silver_tmp["date"] = pd.to_datetime(df_silver_tmp["date"])
        df_silver_tmp = df_silver_tmp.set_index("date")
        df_silver_tmp = df_silver_tmp.loc[start:end]

    # 金価格
    df = df_stock.copy()
    df["date"] = pd.to_datetime(df["date"])
    df = df.set_index("date")
    df = df.loc[start:end]


    # インデックスを揃えて結合
    if df_sp is not None:
        df["MA5_SP"] = df_sp_tmp["ma5"].reindex(df.index)
        df["MA25_SP"] = df_sp_tmp["ma25"].reindex(df.index)
    if df_gold is not None:
        df["MA5_GOLD"] = df_gold_tmp["ma5"].reindex(df.index)
        df["MA25_GOLD"] = df_gold_tmp["ma25"].reindex(df.index)
    if df_silver is not None:
        df["MA5_SILVER"] = df_silver_tmp["ma5"].reindex(df.index)
        df["MA25_SILVER"] = df_silver_tmp["ma25"].reindex(df.index)

    show_plot = ShowPlot()
    fig = show_plot.create_basic_chart(
        df=df.reset_index(),
        code=code,
        name=name,
        start=start,
        end=end
    )
    # ★ 2つのY軸を定義（左：HYMC、右：SP500 & GOLD）
    fig.update_layout(
        yaxis=dict(
            title=f"{name} Price",
            side="left"
        ),
        yaxis2=dict(
            title="SP500 / GOLD",
            overlaying="y",
            side="right"
        )
    )

    # --- SP500（右軸） ---
    if df_sp is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SP"],
                name="SP_MA5",
                line={"color": "blue", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SP"],
                name="SP_MA25",
                line={"color": "gray", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- GOLD（右軸） ---
    if df_gold is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_GOLD"],
                name="GOLD_MA5",
                line={"color": "orange", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_GOLD"],
                name="GOLD_MA25",
                line={"color": "yellow", "width": 1.2},
                yaxis="y2"   # ★ 右軸
            )
        )

    # --- SILVER（左軸） ---
    if df_silver is not None:
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA5_SILVER"],
                name="SILVER_MA5",
                line={"color": "gray", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df.index,
                y=df["MA25_SILVER"],
                name="SILVER_MA25",
                line={"color": "black", "width": 1.2},
                yaxis="y"   # ★ 左軸
            )
        )

    return fig

In [7]:
name = "Americas Gold and Silver"
start = dt.datetime(2025, 1, 1).strftime("%Y-%m-%d")
end = dt.datetime(2026, 4, 17).strftime("%Y-%m-%d")
# グラフ領域の作成
fig = stock_prices_and_gold_prices(
    code=apm,
    name=name,
    start=start,
    end=end,
    df_sp=None,
    df_gold=None,
    df_silver=None
)
fig.show()

取得件数: 531


In [8]:
#### 財務データ

In [9]:
response = request_api.update_corp_finance_data(
    code=apm,
    market=market
)
response

{'result': True}

In [10]:
apm_financials_data = request_api.get_corp_financials_data(code=apm, market=market)
apm_balance_sheet_data = request_api.get_corp_balance_sheet_data(code=apm, market=market)
apm_cash_flow_data = request_api.get_corp_cash_flow_data(code=apm, market=market)
apm_earnings_data = request_api.get_corp_earnings_data(code=apm, market=market)
apm_quarterly_earnings_data = request_api.get_corp_quarterly_earnings_data(code=apm, market=market)

In [11]:
# ４年分の財務データ
apm_financials_data_df = pd.DataFrame(apm_financials_data['results'])
# ４年分のバランスシート
apm_balance_sheet_data_df = pd.DataFrame(apm_balance_sheet_data['results'])
# ４年分のキャッシュフロー
apm_cash_flow_data_df = pd.DataFrame(apm_cash_flow_data['results'])
# ４年分の収益データ
apm_earnings_data_df = pd.DataFrame(apm_earnings_data['results'])
# ４年分の四半期収益データ
apm_quarterly_earnings_data_df = pd.DataFrame(apm_quarterly_earnings_data['results'])

In [12]:
"""
◆ 1. 株価・市場データ
• 現在株価（Price）
• 時価総額（Market Cap）
• 出来高（Volume）
• 52週高値・安値
• Beta（ボラティリティ指標）ß
"""
stock_prices_market_data.stock_prices_and_market_data(
    code=apm,
    market=market,
    bs_df=apm_balance_sheet_data_df
)

取得件数: 458
取得件数: 461
取得件数: 457
取得件数: 460
取得件数: 457
取得件数: 458
取得件数: 458
取得件数: 459
取得件数: 458
取得件数: 458


,close,market_cap,shares_outstanding,higher_rate_par_52_weeks,lower_rate_par_52_weeks,beta
0,0.18,NaN,NaN,2.03,0.125,0.673173
1,0.24,3.792786e+07,158032756.0,2.30,0.240,0.829025
2,1.86,2.914696e+08,156704102.0,2.18,0.530,0.348886
3,0.82,1.228396e+08,149804427.0,2.18,0.530,0.549195
4,0.74,1.108379e+08,149780936.0,10.85,0.680,1.366165


In [13]:
"""
◆ 2. 財務データ（Financials）+ EPS（Earnings Per Share）+ PBR（Price-to-Book Ratio）
• 売上高（Revenue）
• 営業利益（Operating Income）
• 純利益（Net Income）
• EBITDA（企業による）
• 総資産（Total Assets）
• 総負債（Total Liabilities）
• 現金（Cash）
• 希釈EPS（Diluted EPS）
• 基本EPS（Basic EPS）
• 営業キャッシュフロー（Operating Cash Flow）
• フリーキャッシュフロー（Free Cash Flow）
"""
financial_df = financial.calc_financial(
    code = apm,
    market = market,
)
financial_df

取得件数: 2042


,date,revenue,earnings,total_assets,total_debt,cash_and_cash_equivalents,EBITDA,operating_income,basic_eps,diluted_eps,operating_cash_flow,free_cash_flow
0,2021-12-31,NaN,NaN,NaN,NaN,NaN,10388000.0,NaN,NaN,NaN,NaN,NaN
1,2022-12-31,108049000.0,-10091000.0,133857000.0,NaN,80729000.0,-2677000.0,-10217000.0,-0.06,-0.06,-2740000.0,-4944000.0
2,2023-12-31,125324000.0,34761000.0,259914000.0,47458000.0,64907000.0,45032000.0,756000.0,0.22,0.20,9308000.0,3291000.0
3,2024-12-31,254000000.0,19224000.0,305118000.0,70317000.0,62441000.0,55049000.0,40498000.0,0.13,0.12,56638000.0,34525000.0
4,2025-12-31,359827000.0,118159000.0,433932000.0,45076000.0,79211000.0,174262000.0,115245000.0,0.79,0.78,91736000.0,59024000.0


In [14]:
# PBR（Price-to-Book Ratio）やROE（Return on Equity）などの投資指標を計算
financial.calc_stock_investment_indicators(code=apm, market=market)

取得件数: 457


,date,EV,reason,BPS,PBR,ROE,operating_income,basic_eps,diluted_eps
0,2021-12-31,NaN,no_price,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-12-31,NaN,no_price,NaN,NaN,-0.106866,-10217000.0,-0.06,-0.06
2,2023-12-31,NaN,no_price,NaN,NaN,0.271168,756000.0,0.22,0.20
3,2024-12-31,NaN,NaN,0.960339,1.296418,0.133627,40498000.0,0.13,0.12
4,2025-12-31,NaN,NaN,1.760224,5.845847,0.448170,115245000.0,0.79,0.78


### 決算資料を取得

In [15]:
#md_file_path =pdf_to_md.pdf_url_to_markdown(
#    pdf_url="https://www.ayagoldsilver.com/_resources/financials/2025/AIF-2025.pdf",
#    directory_path="/workspace/data",
#)
#md_file_path

In [ ]:

md_file_path = webpage_to_markdown.webpage_to_markdown(
    url = 'https://finance.yahoo.com/news/andean-precious-metals-files-ni-222100100.html',
    directory_path = '/workspace/data'
)
md_file_path

PosixPath('/workspace/data/kiniksa-pharmaceuticals-international-knsa-29-011819245.html.md')